## Dask-on-ray

https://docs.ray.io/en/latest/ray-more-libs/dask-on-ray.html

Ray maintains a Dask compatibility layer for using Ray’s scheduler with Dask-produced Task Graphs.

### Goals

1. Verify whether dask-on-ray works out of the box on “common lsdb workflows”.
2. Compare qualitative experience of using Dask’s dashboard against Ray’s.
3. Compare quantitative performance of Dask client performance against dask-on-ray, both timing and peak memory ideally.
4. If seeing instances where Ray seems valuable, capture in some kind of notebook for the docs.

### Desired outcomes

Some advantages for us/users?
- A tutorial notebook that shows some basic usage patterns where Dask-on-ray has performance/memory advantages.

No advantages?
- A notebook showing what we tried with Dask-on-ray and the lack performance/stability/memory efficiency that led us to that conclusion.

In [1]:
import lsdb

### Version support

Version support is the following according to the documentation:

<img src="./assets/version-support.png" width=600/>

It looks outdated, so we will use the latest package versions unless we find problems.

In [2]:
lsdb.show_versions()


--------      SYSTEM INFO      --------
python        : 3.13.12
python-bits   : 64
OS            : Linux
OS-release    : 5.14.0-611.30.1.el9_7.x86_64
Version       : #1 SMP PREEMPT_DYNAMIC Fri Feb 13 17:04:55 UTC 2026
machine       : x86_64
processor     : x86_64
byteorder     : little
LC_ALL        : 
LANG          : C.UTF-8
--------   INSTALLED VERSIONS   --------
lsdb          : 0.9.0
hats          : 0.9.0
nested-pandas : 0.6.8
pandas        : 2.3.3
numpy         : 2.4.2
dask          : 2026.3.0
pyarrow       : 23.0.1
fsspec        : 2026.3.0


### How to install

Include the default extras to install the dependencies required for the Ray dashboard:

In [ ]:
%pip install -U ray[default] ipywidgets

### Basic usage

We can plan our Dask operations as usual:

In [ ]:
# Select first 10 partitions for testing
gaia = lsdb.open_catalog('https://data.lsdb.io/hats/gaia_dr3').partitions[:10]

Then we just wrap the compute call on a dask-on-ray context:

In [ ]:
import ray
from ray.util.dask import enable_dask_on_ray, disable_dask_on_ray

# Details on how to configure the cluster in notebook 3.
ray.init()

# Wrap compute in a ray context:
enable_dask_on_ray()
gaia.partitions[:10].compute()
disable_dask_on_ray()

""" 
Or using a context manager...
with enable_dask_on_ray():
    gaia.partitions[:10].compute()
"""

# Close the cluster.
ray.shutdown()

2026-04-06 09:32:09,157	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8266 


Computing Catalog:   0%|          | 0/31 [00:00<?, ?it/s]

/astro/users/smcampos/.conda/envs/lsdb/lib/python3.13/site-packages/dask/config.py:835: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(


### Notes:

- Ray does not execute Dask tasks, so our Dask tasks need to be converted into Ray tasks.
- Ray seems to report 31 tasks (~3 tasks per partitions), as well as Dask.
- The tqdm progress bar displays 100% immediately, and before the computation is completed. Maybe it's when all Ray tasks are submitted?

### Checking on progress

With the help of the dashboard we can see how many Ray tasks were submitted and their progress:

<img src="assets/dashboard-jobs.png" width=800>

In `notebook 2.` we will delve into the dashboard a bit more.